# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step workflow for loading and exploring the [FAIR^2 mass spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and adheres to the [ML Commons Croissant standard](https://mlcommons.org/croissant/).

- Title: **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells**
- Description: Analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples, including fields such as accession, description, coverage percentage, peptide counts, molecular weight, calculated pI, and normalized abundances across samples.
- Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields (columns), and their `@id` values for precise referencing.

**Note:** In Croissant, each record set and field/column is uniquely identified by its `@id`.

Let's list out the record sets:

In [ ]:
# List available record sets and their fields, with @id references
record_sets = list(metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}")
for i, rs in enumerate(record_sets):
    print(f'Record Set #{i+1}')
    print(f"  @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print("  Fields (columns):")
    for field in rs.fields:
        print(f"    - @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
    print("-")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All key references (record set and field) are by their `@id`.

**Tip:** Choose the relevant record set (most datasets have one major record set for main tabular data) and use the printed list from the previous step.

In [ ]:
# Example: Use the first record set found
dataframes = {}
record_sets = list(metadata.record_sets)
record_set_ids = [rs.id for rs in record_sets]

# Load all record sets into pandas dataframes
for record_set in record_sets:
    records = list(dataset.records(record_set=record_set.id))
    df = pd.DataFrame(records)
    dataframes[record_set.id] = df

# For demonstration, show the columns and preview for the primary record set
primary_record_set_id = record_sets[0].id if record_sets else None
if primary_record_set_id:
    print(f"Columns in record set {primary_record_set_id}:\n", dataframes[primary_record_set_id].columns.tolist())
    display(dataframes[primary_record_set_id].head())
else:
    print("No record sets available in the dataset.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

Replace `<numeric_field_id>` and `<group_field_id>` with fields listed above (referenced by their `@id`), e.g., for protein mass or abundance, use the matching field `@id`.

In [ ]:
# Choose one numeric field (e.g. molecular_weight) and one categorical group (e.g. description or protein_family)
# Replace with the correct field @ids from the overview— below are typical placeholder examples
record_set_id = primary_record_set_id
df = dataframes[record_set_id]

# List the available columns for selection
print(f"Available fields (columns) in record set {record_set_id}:")
for col in df.columns:
    print(f"- {col}")

# Example field names (update to match the actual @id of the numeric field, e.g. 'molecular_weight')
numeric_field_id = None
for col in df.columns:
    if 'weight' in col.lower() or 'abundance' in col.lower() or 'count' in col.lower() or 'coverage' in col.lower():
        numeric_field_id = col
        break

if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered to records where {numeric_field_id} > {threshold:.2f}")

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by likely categorical field (e.g., description or accession)
        group_field_id = None
        for cand in ['description', 'protein_family', 'accession', 'id']:
            for col in df.columns:
                if cand in col.lower() and col != numeric_field_id:
                    group_field_id = col
                    break
            if group_field_id:
                break

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
    else:
        print(f"Field {numeric_field_id} could not be interpreted as numeric for EDA.")
else:
    print("No suitable numeric field was found. Please revise field selection in EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. Update the field `@id`s as appropriate.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Numeric field histogram
if numeric_field_id is not None and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Example: Boxplot by group field if group_field_id is set
if 'group_field_id' in locals() and group_field_id and group_field_id in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(12,4))
    # Show up to 10 groups only for clarity
    selected = df[group_field_id].value_counts().index[:10]
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df[df[group_field_id].isin(selected)])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

- You have loaded and explored a mass spectrometry protein dataset using its Croissant metadata schema and the `mlcroissant` API.
- You can now further analyze protein expression, modifications, abundance distributions, or relate them to protein categories.
- For customized analyses, reference fields and record sets via their `@id` as shown, and extend this workflow to advanced visualizations, ML modeling, or data transformation tasks.

*This notebook demonstrates how FAIR metadata and the Croissant standard simplify programmatic biomedical data science workflows.*